<a href="https://colab.research.google.com/github/christo444/Dark-Guard/blob/main/Model_p2v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
DarkGuard Phase 2 v3 - multi-class dark pattern classifier (RoBERTa-large)
(v3 = v2 + rewritten synthetic Sneaking lists + 37 targeted hard negatives + a held-out
 backend-phrase check. Parent script v2 md5 a89fa40058cf36881fb2cbdab68dfa48.)

Run in Google Colab (T4 GPU):   %run darkguard_phase2_v2.py
(or paste the whole file into one cell). No file upload needed - the public
dataset is downloaded directly. Total runtime is roughly 20 minutes.

What it does
  1. Builds the training set: real data + synthetic examples (each tagged with a
     `source`, so real vs synthetic performance can be reported separately).
  2. 5-fold stratified CV, grouped by page_id (no page appears in two folds).
  3. Pooled out-of-fold report: overall, REAL-only, per-source, binary-derived,
     and a confidence-calibration table.
  4. Style-diverse OOD holdout (never trained on during CV) + optional BLIND_TESTS.
  5. Trains the final model on everything, writes model card + predict.py, zips.
"""
import os, sys, json, gc, time, subprocess, datetime
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

DATA_URL = "https://raw.githubusercontent.com/yamanalab/ec-darkpattern/master/dataset/dataset.tsv"
SEED, MAX_LENGTH, N_SPLITS = 42, 64, 5
NOT_DARK = "Not Dark Pattern"

# ----------------------------------------------------------------------------
# OPTIONAL: your own blind test. Write examples YOURSELF (not with an AI), then
# list them as ("text", "Category"). They are never trained on during CV, so they
# are the most independent check available. Valid categories: Forced Action,
# Misdirection, Not Dark Pattern, Obstruction, Scarcity, Sneaking, Social Proof,
# Urgency.
# ----------------------------------------------------------------------------
BLIND_TESTS = [
    # ("You must accept marketing emails to continue.", "Forced Action"),
]

# ------------------------- Phase 1 negatives (rounds 1-3) -------------------
PHASE1_NEGATIVES = [
    "Price: $24.99","Price: ₹3,499","$49.00","₹899","Rs. 1,250.00","Total: $128.45","Subtotal: $56.20",
    "$12.99 per item","MRP: ₹2,999","Regular price $75.00","Quantity: 1","Quantity:","Qty: 2","Item quantity",
    "Select quantity","Update quantity","1 item in cart","3 items in your bag","Friday, 25 September",
    "Order placed on March 3, 2024","Delivery by June 12","Estimated arrival: 2-3 business days",
    "Return window: 30 days","Available from January 15","Ships on Monday","Delivering to Dindigul 624005",
    "Ship to: New York, NY 10001","123 Main Street, Apt 4B","Delivery address","Change delivery address",
    "Shipping to United States","Estimated delivery to your area","Order #48213",
    "Tracking number: 1Z999AA10123456784","Invoice #2024-0091","Account number ending in 4417",
    "Reference ID: 88213","30% off","50% discount","Save 20%","10% cashback on this order","Tax: 8.5%",
    "Interest rate: 4.9% APR","4.5 out of 5 stars","Rated 4.2 based on 310 reviews","3 reviews",
    "128 customers rated this 5 stars","Size: 8 UK","Weight: 2.5 kg","Dimensions: 12 x 8 x 4 inches",
    "Available in sizes 6, 8, 10, 12","500ml bottle","Pack of 3","Call us at 1-800-555-0199",
    "Customer service: (555) 123-4567","Page 2 of 14","Showing 1-24 of 350 results","12 items per page",
    "Card ending in 4417","Expires 08/27","3 installments of $16.33","Pay in 4 interest-free payments",
    "Showing 25-48 of 512 results","Displaying 1-20 of 87 items","1-12 of 64 results","Page 1 of 25",
    "View all 214 results","Next page","Ordered on April 18, 2023","Purchased on 12 May 2022",
    "Your order was placed on November 9","Delivered on September 14","Return requested on July 1",
    "Estimated delivery: October 5-7","2 items in your cart","5 items in your bag","You have 4 items saved",
    "0 items in cart","Cart (3)","Wishlist (7)","Only 30% off","Only 15% off select items","Only $5 off",
    "Only ₹200 off","Only 2% cashback","Sold by Amazon","Sold by ThirdPartySeller Inc.",
    "Sold and shipped by Walmart","Fulfilled by Amazon","Subscribe for $9.99/month. Cancel anytime.",
    "Start your 7-day free trial today.","No commitment. Cancel before billing.",
    "Your 14-day free trial begins today.","Try free for 30 days. Cancel anytime.",
    "Billed monthly. Cancel whenever you like.","Book now","Reserve now","Order today","Schedule now",
    "3400","8756","129","45021","SKU 30219","Item #4471",
]

# ------------------------- Phase 2 v1 rare-category examples ----------------
P2V1 = {
 "Forced Action": [
    "You must refer 3 friends before you can claim your discount.","Complete a survey to unlock the checkout button.",
    "Add a protection plan to proceed with your purchase.","You need to create an account before you can complete checkout.",
    "Watch this video before you can close this window.","You must download our app to redeem this coupon.",
    "Rate this product before you can access your order history.","Buy one more item to qualify for free shipping on this order.",
    "You need to upgrade your plan to unlock this feature.","Share this deal on social media to reveal the discount code.",
    "You must accept the terms of our partner offer to continue.","Enter your phone number to proceed with payment.",
    "You need to enable notifications to complete sign-up.","Add a warranty to finish placing your order.",
    "You must subscribe to our SMS list before checkout is enabled.","Verify your email before you can add items to your cart.",
    "You need to invite 2 contacts to unlock free shipping.","Play this mini-game to receive your discount code.",
    "You must select a subscription tier to view pricing.","Sign up for a free consultation to see the final price."],
 "Sneaking": [
    "A $2 donation shows up automatically in your cart total.",
    "Travel insurance ($14.99) is pre-selected on your booking.","Priority support has been included in your order.",
    "Express shipping is automatically selected for you at checkout.","Your total now shows a service fee you did not choose.",
    "A matching accessory appears in your bag without you adding it.","A recommended add-on was placed in your cart for you.",
    "Extended warranty ($9.99) is pre-ticked by default.","The premium package is pre-selected unless you change it.",
    "A carbon offset fee is automatically applied to your order.","A tip is pre-filled in your checkout total.",
    "Trip protection is ticked by default on your reservation.","This subscription box adds a bonus item at extra cost.",
    "A table reservation fee is charged unless you opt out.","A processing fee is applied to your final total.",
    "Your cart contains a recommended bundle item you didn't pick.","A convenience fee is automatically applied to your order.",
    "The recommended upgrade is switched on for your plan.","Your basket quietly includes a $3 charity contribution.",
    "After the first year, this subscription renews automatically with an annual fee."],
 "Obstruction": [
    "To cancel, please call our support line during business hours.","Cancellation requests must be submitted by mail.",
    "You must speak to a retention specialist before cancelling.","Please contact your account manager to cancel your subscription.",
    "Cancellation is only available via live chat, Monday-Friday.","To unsubscribe, please fax your request to our office.",
    "Your account cannot be closed online; please call customer service.","Log out and log back in three times to access cancellation settings.",
    "Cancellation requires a 30-day written notice by post.","You'll need to answer a short survey before we process your cancellation.",
    "This subscription can only be cancelled by the account's original creator.","To delete your account, please email our support team directly.",
    "Please wait on hold to speak with an agent to cancel your plan.","Cancellation option is disabled until your current billing cycle ends.",
    "You must downgrade twice before you can fully cancel your account."],
}

# ------------------------- Phase 2 v2: style-matched + hard negatives -------
P2V2 = {
 "Forced Action": [
    # paper-style precondition (must do/buy something to get the benefit)
    "Free shipping applies only when you add a second item to your order.","Spend $25 more to unlock your discount.",
    "A free trial requires you to sign up for a paid annual plan.","This discount is applied only if you also purchase the extended warranty.",
    "Add any accessory to your cart to receive your free gift.","Enter a valid credit card to start your free trial.",
    "Buy 2 to get 20% off.","To claim your free gift, subscribe to our monthly plan.",
    "Your coupon is valid only with the purchase of a second item.","Subscribe to a premium membership to redeem this offer.",
    # real-style bundled consent (Terms + marketing in ONE agreement)
    "I agree to the Terms of Service and to receive marketing emails from Harbor & Pine.",
    "I accept the Terms & Conditions and agree to receive exclusive offers and promotions from Luma Outdoors.",
    "I have read and agree to the Terms of Use and authorize Vista Home to contact me by phone, text or email with promotions.",
    "Yes, I want to join the VIP Club, and I agree to the Terms and Conditions and to receive emails and special offers.",
    "I agree to the Privacy Policy and terms of use and to receive marketing communications from Northwind Apparel.",
    "I accept the program Terms & Conditions and agree to receive special offers and general promotions from Orbit Mobile.",
    "I've read and accept the terms & conditions. I also authorize Clearview Dental to contact me by text or email about appointments, products and promotions.",
    "I agree to the Terms and Conditions and consent to receive automated marketing texts at the number provided.",
    "I accept the membership Terms and agree to receive newsletters and third-party promotional offers.",
    "I agree to the Terms of Sale and to receive order updates and special promotional offers."],
 "Sneaking": [
    # terse add-ons (real style: 'Purchase protection added', 'Shipping Insurance Remove Insurance')
    "Shipping protection added","Order protection: $2.99 Remove","Donation of $1.00 added to your order",
    "Parcel Insurance Remove","Care package $4.99 - click to remove","Extended service plan: $12.00 Remove",
    "Priority processing $4.99 (selected)","Roundup donation of $0.50 applies at checkout",
    "Seat selection fee applies to your booking","Bag protection ($6.00) added",
    # receipts containing unusual fee lines (real style)
    "Merchandise: $54.00 Service Charge: $6.50 Delivery: $9.99 Tax: $4.20 Order Total: $74.69",
    "Subtotal $32.00 Handling & Processing $3.99 Tax $2.71 Total $38.70",
    "Item Total $27.50 Rush Fee $5.00 Tax $2.10 Grand Total $34.60",
    "Order Subtotal $41.99 Care & Handling $4.49 Convenience Fee $2.50 Tax $3.11 Total $52.09",
    "Items $88.00 Service Fee $7.25 Tax $6.16 Amount Due $101.41",
    "Cart Subtotal $19.00 Membership Fee $9.99 Tax $1.52 Total $30.51",
    # memberships bundled into an order (real style: 'Free 60 day VIP trial membership with this order.')
    "Free shipping comes with a 30-day trial membership that starts with this order. Cancel anytime.",
    "Your order starts a 7-day trial of Premium Club, billed monthly after the trial.",
    "A 30-day rewards membership begins with this purchase. Cancel anytime.",
    "This order enrolls you in a subscription that renews automatically."],
 "Obstruction": [
    "To cancel your membership, please call 1-800-555-0142 between 9am and 5pm EST.",
    "Membership cancellations must be made by contacting customer service by phone; online cancellation is not available.",
    "You may cancel at any time by sending written notice to our billing department.",
    "To stop your automatic renewal, email billing@example.com at least 15 days before your renewal date.",
    "Subscriptions can only be cancelled through our customer care team.",
    "To cancel your order, please contact us by phone; we cannot cancel orders online.",
    "Automatic renewals will continue until you notify us by phone.",
    "Cancel your trial by calling us before the trial ends, or you will be charged.",
    "Requests to cancel are processed within 10 business days after we receive your written request.",
    "To end your subscription, complete the cancellation form and mail it to the address below.",
    "You can pause your plan online, but to cancel you must chat with a member of our team.",
    "Cancellation requests are not accepted through the website or app.",
    "To close your account, you must first cancel all active shipments by calling support.",
    "Contact our retention team to discuss cancelling your plan."],
 NOT_DARK: [
    # plain consent / terms (real negatives look like this)
    "I agree to the Terms and Conditions.","By placing your order, you agree to our Terms of Use and Privacy Policy.",
    "I have read and agree to the return policy.","Read our Privacy Policy.",
    "Tick this box if you would like to receive our newsletter. You can unsubscribe at any time.",
    "Marketing emails are optional. Choose whether you would like to receive them.",
    "Email me my order updates only.","By using this site, you accept our use of cookies.",
    # clearly optional add-ons
    "Add gift wrap for $4.99 (optional).","Would you like to add an extended warranty? Select Yes or No.",
    "Optional: add shipping protection for $2.99.","Add a gift message (optional).",
    "Coverage is not included. You can add protection for $8 if you wish.",
    "Add a donation of your choice, or leave this blank.","Choose your delivery speed: Standard or Express.",
    # easy cancellation (counter-weight to the 'cancel' => Obstruction shortcut)
    "Cancel anytime in your account settings.","You can cancel your subscription online in one click.",
    "Manage or cancel your plan from the Account page.","Cancel online, by phone or by email, whichever you prefer.",
    "No cancellation fee. Cancel from your dashboard at any time.",
    "To cancel your order, click Cancel Order on the order details page.",
    "Cancel or pause your membership at any time from your profile.",
    "Cancelling is easy: just select Cancel Plan in Settings.",
    # receipts with no extra fee lines
    "Subtotal $32.00 Tax $2.71 Total $34.71","Item $54.00 Tax $4.20 Order Total $58.20",
    "Merchandise $19.99 Tax $1.60 Total $21.59",
    # neutral account / registration
    "Create an account to track your orders (optional).","Guest checkout available.",
    "Sign in to view your order history.","Enter your email address to receive your receipt.",
    "Phone number (optional)","Verify your email address to secure your account.",
    "Continue without an account.","Already have an account? Sign in."],
}

# ------------------------- Phase 2 v3: targeted hard negatives (37) ---------
# Written to address specific, observed false positives. Deliberately NOT the backend's own
# failing phrases (those live in BACKEND_HELDOUT below and are never trained on).
# Real-data support was checked first (see technical report, new section): the real data has
# almost no benign abundance / self-service-cancel / account-deletion text, so these are
# SYNTHETIC and are tagged source="p2v3_hard_negative" so they can be reported separately.
P2V3_NEGATIVES = [
    # in-stock abundance (no number, no urgency)
    "Plenty in stock.","Well stocked and ready to dispatch.","Available now.","In stock, ships today.",
    "Currently in stock.","In stock and available for immediate dispatch.","Good availability.","Back in stock.",
    # free shipping / delivery without urgency
    "Free delivery on orders above $75.","Shipping is free for orders over $30.","Free standard delivery on all orders.",
    "Standard shipping: free.","Free returns within 30 days.",
    # included / free warranty and guarantee, included accessories
    "Comes with a 2-year warranty at no extra cost.","Two-year manufacturer warranty.",
    "Covered by a 30-day money-back guarantee.","Free 12-month warranty with every purchase.",
    "Warranty: 1 year (standard).","Charger and cable included in the box.",
    "Batteries included.","Prices include VAT.","A gift box is included with every order.",
    # bare numbers with thousands separators
    "2,846","58,913","1,205","7,350","12,480","96,001",
    # clear, self-service cancel / opt-out / account deletion
    "Unsubscribe in one click using the link at the bottom of any email.",
    "You can opt out whenever you like in your notification settings.",
    "Turn off auto-renewal at any time in your billing settings.",
    "Opt out of marketing emails at any time by clicking Unsubscribe.",
    "Cancel your order from the Orders page before it ships.",
    "Delete your account any time from Account > Privacy; your data is erased immediately.",
    "You can close your account whenever you want from your profile settings.",
    "Download your data or delete your account in Privacy Settings.",
    "Deleting your account removes your personal data. You will be asked to confirm.",
]

# ------------------------- Backend report phrases: HELD-OUT check ------------------------
# (text, expected category). Source: the backend team's fine_tuning_recommendations.md. The expected
# labels are the backend's, except where noted. These are asserted to be absent (punctuation-
# insensitively) from every training/OOD row; the nearest training text + similarity is printed.
BACKEND_HELDOUT = [
    ("3,417", NOT_DARK),
    ("In stock and ready to ship", NOT_DARK),
    ("In stock. Ready for standard delivery.", NOT_DARK),
    ("Free standard shipping on orders over $50", NOT_DARK),
    ("Cancel anytime online with 1-click.", NOT_DARK),
    ("You can opt out of this at any time.", NOT_DARK),
    ("Permanently delete your account and all associated data. This action cannot be undone.", NOT_DARK),
    ("1 year manufacturer warranty included", NOT_DARK),
]
# Reported, never scored. "No Cost EMI:" is an out-of-scope regional (India) finance term; it is held-out
# (asserted absent from training) and documented, not fixed.
KNOWN_GAPS = ["No Cost EMI:"]
# "12 rooms available" is a labelling-CONVENTION dispute, not a model bug: the dataset labels numeric
# availability counts Scarcity (the backend expected Not Dark). A near-copy "12 rooms available." is
# deliberately in OOD_ROWS (Scarcity), so this phrase is NOT held-out; it is reported for transparency only.
CONVENTION_DISPUTES = ["12 rooms available"]
REPORT_ONLY = KNOWN_GAPS + CONVENTION_DISPUTES
# Phrases the backend reported as already passing; guards against regressions. These MAY overlap
# training text (e.g. "Add to cart"), so they are informative but not a clean held-out test.
BACKEND_REGRESSION = [
    ("Order number: 994231", NOT_DARK), ("2", NOT_DARK), ("Shop Now", NOT_DARK), ("Add to cart", NOT_DARK),
    ("Hurry up, only 2 left in stock!", "Scarcity"), ("24 people are viewing this right now", "Social Proof"),
    ("Uncheck this box if you DO NOT wish to NOT receive daily promotional emails", "Misdirection"),
    ("After your 7-day free trial, you will automatically be billed $199 annually.", "Forced Action"),
]

# ------------------------- OOD holdout (never trained on during CV) ---------
# NOTE: written by the same author as the training examples (an AI), so this is a
# weaker test than independently-written examples. Use BLIND_TESTS for that.
OOD_ROWS = [
 ("Tick this box to confirm you accept our terms and would like to hear about special offers from us and selected partners.","Forced Action"),
 ("I agree to the Terms of Service and consent to marketing texts from Brightwave Retail.","Forced Action"),
 ("You must add a paid plan to activate your free trial.","Forced Action"),
 ("Spend $40 more to receive your complimentary gift.","Forced Action"),
 ("Registration requires you to opt in to promotional emails.","Forced Action"),
 ("By joining, I accept the membership terms and agree to be contacted about deals by email, phone or SMS.","Forced Action"),
 ("Discount unlocks only after you buy any two items from this collection.","Forced Action"),
 ("I confirm I accept the terms and wish to receive promotional communications.","Forced Action"),
 ("Delivery protection $3.49 added to your basket","Sneaking"),
 ("Order total: Items $41.00 | Service fee $4.50 | Delivery $6.00 | Tax $3.10 | Total $54.60","Sneaking"),
 ("A charity donation of $2 has been included in your order","Sneaking"),
 ("Your basket includes a 30-day trial of Prime Select. Membership fee applies after trial.","Sneaking"),
 ("Gift receipt and packaging added: $2.50","Sneaking"),
 ("Seat upgrade fee included in your fare","Sneaking"),
 ("Recommended: Premium Support (pre-selected)","Sneaking"),
 ("Insurance for your parcel has been added at $5.99","Sneaking"),
 ("To cancel your plan, please phone our team; we don't accept cancellations online.","Obstruction"),
 ("You can only unsubscribe by sending a letter to our head office.","Obstruction"),
 ("Cancelling requires you to speak with a specialist on our support line.","Obstruction"),
 ("Membership can be terminated only by written request posted to our billing office.","Obstruction"),
 ("To stop future shipments, call customer service before the 5th of each month.","Obstruction"),
 ("We cannot process cancellation requests submitted through the app or website.","Obstruction"),
 ("Please chat with an advisor if you would like to end your subscription.","Obstruction"),
 ("Trial cancellations must be made by phone before the trial period ends.","Obstruction"),
 ("By placing this order you accept our Terms of Sale.",NOT_DARK),
 ("Tick to receive our newsletter (optional).",NOT_DARK),
 ("Would you like to add gift packaging for $3? (optional)",NOT_DARK),
 ("You can cancel your membership online at any time from Account Settings.",NOT_DARK),
 ("Subtotal $28.00 Shipping $6.50 Tax $2.30 Total $36.80",NOT_DARK),
 ("Add travel insurance (optional).",NOT_DARK),
 ("Cancel your subscription in two clicks from your profile page.",NOT_DARK),
 ("Phone number (optional)  ",NOT_DARK),
 ("Create an account to speed up future checkouts (optional).",NOT_DARK),
 ("Estimated tax: $2.10",NOT_DARK),
 ("Manage your plan or cancel any time online.",NOT_DARK),
 ("Use code SUMMER for 20% off.",NOT_DARK),
 ("Free shipping on orders over $50",NOT_DARK),
 ("Standard Economy Fare",NOT_DARK),
 ("38 people are viewing this right now.","Social Proof"),
 ("Someone in Chicago just bought this 5 minutes ago.","Social Proof"),
 ("12 rooms available.","Scarcity"),
 ("Save 15% when you buy two or more.","Forced Action"),   # paper-analog: Table 1 labels 'buy 2 get 1 free' Forced Action
]


# Independent from all training/OOD data above - simple, unambiguous cases used only
# to catch a bad training run (e.g. an unlucky loss spike) before the model ships.
SANITY_CHECKS = [
    ("Add to cart", NOT_DARK), ("Checkout", NOT_DARK), ("Thank you for your order.", NOT_DARK),
    ("Only 1 left in stock!", "Scarcity"), ("Offer ends in 10 minutes!", "Urgency"),
    ("500 people bought this today.", "Social Proof"),
]

def _key(t): return " ".join(str(t).lower().split())
def _has_pua(t): return any(0xE000 <= ord(c) <= 0xF8FF for c in str(t))
def _pkey(t): return " ".join("".join(ch if ch.isalnum() else " " for ch in str(t).lower()).split())
def _toks(t): return set(_pkey(t).split())
def _nearest(text, train_texts):
    """(similarity, nearest training text) by token-set Jaccard: a transparency check, not a metric."""
    a, best, bt = _toks(text), 0.0, ""
    for t in train_texts:
        b = _toks(t); u = len(a | b)
        if u and len(a & b) / u > best: best, bt = len(a & b) / u, t
    return best, bt

def _groups(df):
    """Group rows that share a page_id OR the same normalized text, so neither
    same-page rows nor case/space-variant duplicates can straddle train/validation."""
    parent = list(range(len(df)))
    def find(x):
        while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
        return x
    seen = {}
    for i, (p, k) in enumerate(zip(df["page_id"], df["_k"])):
        for key in (("p", p), ("k", k)):
            if key in seen: parent[find(i)] = find(seen[key])
            else: seen[key] = i
    return np.array([find(i) for i in range(len(df))])

def build_dataset(verbose=True):
    real = pd.read_csv(DATA_URL, sep="\t")
    real = real[~real["text"].apply(_has_pua)].copy()
    real["source"] = "real"
    rows, nid = [], 90000
    def add(text, cat, src):
        nonlocal nid
        rows.append({"page_id": nid, "text": text, "label": int(cat != NOT_DARK),
                     "Pattern Category": cat, "source": src}); nid += 1
    for t in PHASE1_NEGATIVES: add(t, NOT_DARK, "p1_negative")
    for cat, ts in P2V1.items():
        for t in ts: add(t, cat, "p2v1_rare")
    for cat, ts in P2V2.items():
        for t in ts: add(t, cat, "p2v2_hard_negative" if cat == NOT_DARK else "p2v2_rare")
    for t in P2V3_NEGATIVES: add(t, NOT_DARK, "p2v3_hard_negative")
    synth = pd.DataFrame(rows)
    real["_k"] = real["text"].map(_key); synth["_k"] = synth["text"].map(_key)
    real_keys = set(real["_k"])
    keep = ~(synth["_k"].isin(real_keys) | synth["_k"].duplicated())
    dropped = int((~keep).sum())                       # only SYNTHETIC rows can be dropped
    synth = synth[keep]
    cols = ["page_id", "text", "label", "Pattern Category", "source", "_k"]
    pool = pd.concat([real[cols], synth[cols]], ignore_index=True)   # every real row is kept
    pool["group"] = _groups(pool)
    keys = set(pool["_k"])
    ood_rows, seen, removed = [], set(), 0
    for i, (t, cat) in enumerate(list(OOD_ROWS) + [(t, c) for t, c in BLIND_TESTS]):
        k = _key(t)
        if k in keys or k in seen: removed += 1; continue
        seen.add(k)
        src = "ood_holdout" if i < len(OOD_ROWS) else "blind_test"
        ood_rows.append({"page_id": 95000 + i, "text": t.strip(), "label": int(cat != NOT_DARK),
                         "Pattern Category": cat, "source": src, "_k": k})
    ood = pd.DataFrame(ood_rows)
    categories = sorted(pool["Pattern Category"].unique())
    assert set(ood["Pattern Category"]) <= set(categories), "Unknown category in OOD/BLIND set"
    l2i = {c: i for i, c in enumerate(categories)}
    pool["category_id"] = pool["Pattern Category"].map(l2i)
    ood["category_id"] = ood["Pattern Category"].map(l2i)
    pool = pool.drop(columns="_k"); ood = ood.drop(columns="_k")
    held = {_pkey(t) for t, _ in BACKEND_HELDOUT} | {_pkey(t) for t in KNOWN_GAPS}
    train_pk = {_pkey(t) for t in pool["text"]} | {_pkey(t) for t in ood["text"]}
    clash = held & train_pk
    assert not clash, f"Held-out backend phrases leaked into training/OOD data: {sorted(clash)}"
    if verbose:
        print(f"Pool: {len(pool)} rows = {int((pool.source=='real').sum())} real (all kept) + {int((pool.source!='real').sum())} synthetic ({dropped} synthetic dropped as duplicates) | OOD/blind holdout: {len(ood)} ({removed} removed: already in training)")
        print("\nCategory x source:")
        print(pd.crosstab(pool["Pattern Category"], pool["source"], margins=True).to_string())
        real_n = pool[pool.source == "real"]["Pattern Category"].value_counts()
        print("\nREAL examples per category (this is the honest sample size):", real_n.to_dict())
    return pool, ood, categories

def make_folds(pool, verbose=True):
    sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    fold = np.full(len(pool), -1)
    for f, (_, va) in enumerate(sgkf.split(pool["text"], pool["category_id"], groups=pool["group"])):
        fold[va] = f
    assert (fold >= 0).all()
    if verbose:
        ct = pd.crosstab(pool["Pattern Category"], fold)
        print("\nValidation rows per class per fold:\n" + ct.to_string())
        assert (ct.values > 0).all(), "A class is missing from some fold"
        k = pool["text"].map(_key)
        for f in range(N_SPLITS):   # no page and no near-duplicate text in two folds
            assert set(pool.loc[fold == f, "page_id"]).isdisjoint(set(pool.loc[fold != f, "page_id"]))
            assert set(k[fold == f]).isdisjoint(set(k[fold != f]))
        print("Verified: no page_id and no (case/space-normalized) text appears in both train and validation of any fold.")
    return fold

def softmax(z):
    z = z - z.max(axis=1, keepdims=True); e = np.exp(z); return e / e.sum(axis=1, keepdims=True)

def _binary(true_ids, pred_ids, categories):
    nd = categories.index(NOT_DARK)
    t, p = np.array(true_ids) != nd, np.array(pred_ids) != nd
    tp, fp, fn, tn = (t & p).sum(), (~t & p).sum(), (t & ~p).sum(), (~t & ~p).sum()
    acc = (tp + tn) / len(t); pr = tp / max(tp + fp, 1); rc = tp / max(tp + fn, 1)
    return acc, pr, rc, (2 * pr * rc / max(pr + rc, 1e-9)), fn, fp

def report_oof(pool, probs, categories):
    pred = probs.argmax(1); true = pool["category_id"].values; K = len(categories)
    out = {}
    print("\n" + "=" * 78 + "\nPOOLED OUT-OF-FOLD RESULTS (every row predicted by a model that never trained on it)\n" + "=" * 78)
    print(f"\n[A] ALL rows - accuracy {accuracy_score(true, pred):.4f}")
    print(classification_report(true, pred, labels=range(K), target_names=categories, zero_division=0, digits=3))
    m = (pool["source"] == "real").values
    print(f"[B] REAL rows only ({m.sum()} rows) - the honest number. accuracy {accuracy_score(true[m], pred[m]):.4f}")
    rep = classification_report(true[m], pred[m], labels=range(K), target_names=categories, zero_division=0, digits=3, output_dict=True)
    print(classification_report(true[m], pred[m], labels=range(K), target_names=categories, zero_division=0, digits=3))
    out["real_macro_f1"] = rep["macro avg"]["f1-score"]; out["real_accuracy"] = accuracy_score(true[m], pred[m])
    out["real_per_class"] = {c: (rep[c]["precision"], rep[c]["recall"], rep[c]["f1-score"], int(rep[c]["support"])) for c in categories}
    print("[C] Accuracy by data source (large real-vs-synthetic gaps expose template effects):")
    g = pd.DataFrame({"source": pool["source"], "ok": pred == true}).groupby("source")["ok"].agg(["mean", "count"])
    print(g.rename(columns={"mean": "accuracy", "count": "rows"}).round(3).to_string())
    print("\n[D] Confusion matrix, REAL rows (rows=true, cols=predicted):")
    cm = confusion_matrix(true[m], pred[m], labels=range(K))
    print(pd.DataFrame(cm, index=categories, columns=[c[:7] for c in categories]).to_string())
    print("\n[E] Dark-vs-not derived from the category prediction (compare with Phase 1 binary ~98%):")
    for name, mm in (("all rows", np.ones(len(true), bool)), ("real only", m)):
        acc, pr, rc, f1, fn, fp = _binary(true[mm], pred[mm], categories)
        print(f"    {name:9s}: acc {acc:.4f} | precision {pr:.4f} | recall {rc:.4f} | F1 {f1:.4f} | missed dark patterns {fn} | false alarms {fp}")
        out["binary_" + name.replace(" ", "_")] = (acc, pr, rc, f1)
    print("\n[F] Confidence calibration (what to trust / what to flag as uncertain in the backend):")
    conf = probs.max(1); bins = [0, .6, .8, .95, 1.0001]
    rows = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        s = (conf >= lo) & (conf < hi)
        rows.append({"confidence": f"{lo:.2f}-{min(hi,1):.2f}", "rows": int(s.sum()), "accuracy": round(float((pred[s] == true[s]).mean()), 3) if s.any() else None})
    print(pd.DataFrame(rows).to_string(index=False))
    return out

def report_ood(ood, probs, categories, per_fold_acc=None):
    pred = probs.argmax(1); true = ood["category_id"].values
    print("\n" + "=" * 78 + "\nFRESH OOD HOLDOUT (never in training; 5-fold-model ensemble)\n" + "=" * 78)
    for (_, r), p, pr in zip(ood.iterrows(), pred, probs):
        ok = "OK " if p == r.category_id else "MISS"
        print(f"{ok} expected={r['Pattern Category']:<16} predicted={categories[p]:<16} conf={pr.max():.2f} | {r.text[:80]}")
    for src in ood["source"].unique():
        s = (ood["source"] == src).values
        print(f"\n{src}: category accuracy {np.mean(pred[s] == true[s]):.3f} on {s.sum()} examples")
    acc, pr, rc, f1, fn, fp = _binary(true, pred, categories)
    print(f"Dark-vs-not on OOD: acc {acc:.3f} | recall {rc:.3f} | precision {pr:.3f}")
    if per_fold_acc: print(f"Single-fold-model OOD accuracy: mean {np.mean(per_fold_acc):.3f}, std {np.std(per_fold_acc):.3f} (run-to-run noise)")

def report_backend(title, rows, probs, categories, train_texts):
    """rows = [(text, expected)]; prints OK/MISS, confidence, and nearest-training-text similarity."""
    print("\n" + "=" * 78 + f"\n{title}\n" + "=" * 78)
    pred = probs.argmax(1); n_ok = 0; res = []
    for (t, exp), p, pr in zip(rows, pred, probs):
        ok = categories[p] == exp; n_ok += ok
        sim, near = _nearest(t, train_texts)
        flag = "  <-- near-duplicate of training text" if sim >= 0.6 else ""
        print(f"{'OK  ' if ok else 'MISS'} expected={exp:<16} predicted={categories[p]:<16} conf={pr.max():.2f} | {t[:60]}")
        print(f"       nearest training text (Jaccard {sim:.2f}): {near[:70]!r}{flag}")
        res.append({"text": t, "expected": exp, "predicted": categories[p], "confidence": round(float(pr.max()), 3),
                    "nearest_train_jaccard": round(sim, 2), "correct": bool(ok)})
    print(f"Result: {n_ok}/{len(rows)} correct (small, hand-picked set: indicative only, not an accuracy estimate)")
    return res

def report_gaps(title, texts, probs, categories):
    print("\n" + "=" * 78 + f"\n{title}\n" + "=" * 78)
    for t, pr in zip(texts, probs):
        top2 = np.argsort(-pr)[:2]
        print(f"INFO  {t!r}: {categories[top2[0]]} ({pr[top2[0]]:.2f}), then {categories[top2[1]]} ({pr[top2[1]]:.2f})")

PREDICT_PY = r'''"""DarkGuard classifier - usage:  python predict.py "Only 2 left in stock!" "Add to cart" """
import sys, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

class DarkGuardClassifier:
    def __init__(self, model_dir=".", device=None, max_length=64):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.tok = AutoTokenizer.from_pretrained(model_dir)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(self.device).eval()
        self.max_length = max_length
        self.id2label = {int(k): v for k, v in self.model.config.id2label.items()}

    @torch.no_grad()
    def classify(self, texts, min_confidence=0.0, batch_size=32):
        """Returns one dict per text: category, is_dark_pattern, confidence, second_choice, uncertain."""
        if isinstance(texts, str): texts = [texts]
        out = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            enc = self.tok(batch, padding=True, truncation=True, max_length=self.max_length, return_tensors="pt").to(self.device)
            probs = torch.softmax(self.model(**enc).logits.float(), dim=-1).cpu()
            for text, p in zip(batch, probs):
                top = torch.topk(p, k=2)
                cat, conf = self.id2label[int(top.indices[0])], float(top.values[0])
                out.append({"text": text, "category": cat, "is_dark_pattern": cat != "Not Dark Pattern",
                            "confidence": round(conf, 4),
                            "second_choice": {"category": self.id2label[int(top.indices[1])], "confidence": round(float(top.values[1]), 4)},
                            "uncertain": conf < min_confidence})
        return out

if __name__ == "__main__":
    clf = DarkGuardClassifier(".")
    for r in clf.classify(sys.argv[1:] or ["Only 2 left in stock!", "Add to cart"]):
        print(r)
'''

def main():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers"], check=False)
    import torch, transformers
    from transformers import AutoTokenizer, RobertaForSequenceClassification, TrainingArguments, Trainer
    try:
        from transformers.utils import logging as hf_logging; hf_logging.set_verbosity_error()
    except Exception: pass
    print("torch", torch.__version__, "| transformers", transformers.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (enable a T4 GPU!)")
    if not torch.cuda.is_available(): raise SystemExit("No GPU: Runtime > Change runtime type > T4 GPU")

    pool, ood, categories = build_dataset()
    K = len(categories); id2label = dict(enumerate(categories)); label2id = {c: i for i, c in id2label.items()}
    fold = make_folds(pool)
    tok = AutoTokenizer.from_pretrained("roberta-large")
    enc = lambda texts: tok(list(texts), padding="max_length", truncation=True, max_length=MAX_LENGTH)

    class DS(torch.utils.data.Dataset):
        def __init__(self, texts, labels): self.e, self.y = enc(texts), list(labels)
        def __len__(self): return len(self.y)
        def __getitem__(self, i):
            d = {k: torch.tensor(v[i]) for k, v in self.e.items()}; d["labels"] = torch.tensor(int(self.y[i])); return d

    class WeightedTrainer(Trainer):
        def __init__(self, *a, class_weights=None, **kw):
            super().__init__(*a, **kw); self.cw = class_weights
        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            labels = inputs.pop("labels"); outputs = model(**inputs)
            loss = torch.nn.functional.cross_entropy(outputs.logits.float(), labels, weight=self.cw.to(outputs.logits.device))
            return (loss, outputs) if return_outputs else loss

    def train_model(train_df, tag, seed_offset=0):
        cw = torch.tensor(compute_class_weight("balanced", classes=np.arange(K), y=train_df["category_id"].values), dtype=torch.float)
        model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=K, id2label=id2label, label2id=label2id)
        args = TrainingArguments(output_dir=f"./p2v3_{tag}", num_train_epochs=4, per_device_train_batch_size=8,
            gradient_accumulation_steps=2, per_device_eval_batch_size=32, learning_rate=2e-5, seed=SEED + seed_offset,
            eval_strategy="no", save_strategy="no", logging_steps=100, report_to="none", disable_tqdm=True, fp16=True)
        tr = WeightedTrainer(model=model, args=args, train_dataset=DS(train_df["text"], train_df["category_id"]), class_weights=cw)
        tr.train(); return model, tr

    ood_ds = DS(ood["text"], ood["category_id"])
    bc_rows = BACKEND_HELDOUT + BACKEND_REGRESSION
    bc_ds = DS([t for t, _ in bc_rows], [label2id[c] for _, c in bc_rows])
    gap_ds = DS(REPORT_ONLY, [0] * len(REPORT_ONLY))          # dummy labels: reported, never scored
    bc_sum = np.zeros((len(bc_rows), K)); gap_sum = np.zeros((len(REPORT_ONLY), K))
    train_texts_cv = list(pool["text"])      # CV models never train on the OOD rows
    oof = np.zeros((len(pool), K)); ood_sum = np.zeros((len(ood), K)); per_fold_ood = []
    for f in range(N_SPLITS):
        t0 = time.time(); tr_m = fold != f; va_m = fold == f
        print(f"\n===== FOLD {f+1}/{N_SPLITS}: train {tr_m.sum()} | validate {va_m.sum()} =====")
        model, trainer = train_model(pool[tr_m], f"fold{f+1}")
        oof[va_m] = softmax(trainer.predict(DS(pool.loc[va_m, "text"], pool.loc[va_m, "category_id"])).predictions)
        p = softmax(trainer.predict(ood_ds).predictions); ood_sum += p
        bc_sum += softmax(trainer.predict(bc_ds).predictions); gap_sum += softmax(trainer.predict(gap_ds).predictions)
        per_fold_ood.append(float((p.argmax(1) == ood["category_id"].values).mean()))
        print(f"fold {f+1} done in {time.time()-t0:.0f}s | fold accuracy {(oof[va_m].argmax(1) == pool.loc[va_m, 'category_id'].values).mean():.4f}")
        del model, trainer; gc.collect(); torch.cuda.empty_cache()

    metrics = report_oof(pool, oof, categories)
    report_ood(ood, ood_sum / N_SPLITS, categories, per_fold_ood)
    nh = len(BACKEND_HELDOUT)
    report_backend("BACKEND HELD-OUT PHRASES (never trained on; 5-fold-model ensemble; CV models did not see OOD either)",
                   BACKEND_HELDOUT, (bc_sum / N_SPLITS)[:nh], categories, train_texts_cv)
    report_backend("BACKEND REGRESSION GUARDS (may overlap training text)",
                   BACKEND_REGRESSION, (bc_sum / N_SPLITS)[nh:], categories, train_texts_cv)
    report_gaps("KNOWN GAP + CONVENTION DISPUTE (reported only)", REPORT_ONLY, gap_sum / N_SPLITS, categories)

    print("\n" + "=" * 78 + "\nFINAL MODEL: training on ALL rows (pool + OOD/blind examples)\n" + "=" * 78)
    full = pd.concat([pool, ood], ignore_index=True)

    for attempt in range(1, 4):
        model, trainer = train_model(full, "final", seed_offset=attempt - 1)
        sc_texts = [t for t, _ in SANITY_CHECKS]
        sc_probs = softmax(trainer.predict(DS(sc_texts, [label2id[c] for _, c in SANITY_CHECKS])).predictions)
        sc_pred = [categories[i] for i in sc_probs.argmax(1)]
        n_ok = sum(p == e for p, (_, e) in zip(sc_pred, SANITY_CHECKS))
        print(f"\nFinal-model sanity check (attempt {attempt}): {n_ok}/{len(SANITY_CHECKS)} passed")
        for (text, exp), p, pr in zip(SANITY_CHECKS, sc_pred, sc_probs):
            print(f"  {'OK  ' if p==exp else 'FAIL'} expected={exp:<16} predicted={p:<16} conf={pr.max():.2f} | {text}")
        if n_ok >= len(SANITY_CHECKS) - 1:      # allow at most 1 miss (genuinely ambiguous cases happen)
            break
        print("Too many sanity checks failed - this training run is unreliable. Retraining the final model...")
        del model, trainer; gc.collect(); torch.cuda.empty_cache()
    else:
        raise SystemExit(f"Final model failed the sanity check {attempt} times in a row. "
                          "Something beyond ordinary run-to-run noise is wrong - stop and report this rather than shipping the model.")

    train_texts_final = list(full["text"])
    bc_final_p = softmax(trainer.predict(bc_ds).predictions)
    bc_final = report_backend("BACKEND HELD-OUT PHRASES - FINAL MODEL (what the backend will actually run)",
                              BACKEND_HELDOUT, bc_final_p[:nh], categories, train_texts_final)
    report_backend("BACKEND REGRESSION GUARDS - FINAL MODEL", BACKEND_REGRESSION, bc_final_p[nh:], categories, train_texts_final)
    report_gaps("KNOWN GAP + CONVENTION DISPUTE - FINAL MODEL", REPORT_ONLY, softmax(trainer.predict(gap_ds).predictions), categories)

    out_dir = "./darkguard_multiclass_v3"; os.makedirs(out_dir + "/evaluation", exist_ok=True)
    model.save_pretrained(out_dir); tok.save_pretrained(out_dir)
    with open(out_dir + "/predict.py", "w") as fh: fh.write(PREDICT_PY)
    full.drop(columns=["category_id"]).to_csv(out_dir + "/evaluation/training_data.tsv", sep="\t", index=False)
    ev = pool[["text", "Pattern Category", "source"]].copy(); ev["fold"] = fold + 1
    ev["predicted_oof"] = [categories[i] for i in oof.argmax(1)]; ev["confidence_oof"] = oof.max(1).round(4)
    ev.to_csv(out_dir + "/evaluation/cv_out_of_fold_predictions.tsv", sep="\t", index=False)
    manifest = {"created": datetime.datetime.now().isoformat(timespec="seconds"), "base_model": "roberta-large",
        "categories": categories, "max_length": MAX_LENGTH, "epochs": 4, "learning_rate": 2e-5, "effective_batch_size": 16,
        "loss": "class-weighted cross-entropy (balanced)", "cv": f"{N_SPLITS}-fold StratifiedGroupKFold by page_id, seed {SEED}",
        "training_rows": len(full), "rows_by_source": full["source"].value_counts().to_dict(),
        "rows_by_category": full["Pattern Category"].value_counts().to_dict(),
        "parent_script_v2_md5": "a89fa40058cf36881fb2cbdab68dfa48", "backend_heldout_final_model": bc_final,
        "versions": {"torch": torch.__version__, "transformers": transformers.__version__},
        "cv_real_only": {"accuracy": round(metrics["real_accuracy"], 4), "macro_f1": round(metrics["real_macro_f1"], 4),
                         "per_class_precision_recall_f1_support": {k: [round(x, 3) if isinstance(x, float) else x for x in v] for k, v in metrics["real_per_class"].items()}}}
    json.dump(manifest, open(out_dir + "/training_manifest.json", "w"), indent=2)
    rows = "\n".join(f"| {c} | {v[0]:.2f} | {v[1]:.2f} | {v[2]:.2f} | {v[3]} |" for c, v in metrics["real_per_class"].items())
    card = f"""# DarkGuard multi-class model (v3)
RoBERTa-large fine-tuned to classify short e-commerce UI text into: {', '.join(categories)}.
`is_dark_pattern = category != "Not Dark Pattern"`.

## Usage
```
pip install torch transformers
python predict.py "Only 2 left in stock!" "Add to cart"
```
Use `DarkGuardClassifier(...).classify(texts, min_confidence=0.95)` from your own code - our own calibration data (see evaluation/) showed accuracy is ~97% above 0.95 confidence but under 65% below it. Each result has `category`,
`is_dark_pattern`, `confidence`, `second_choice` and `uncertain`. Scarcity/Urgency are genuinely ambiguous in the
taxonomy; `second_choice` helps. CPU inference is slow for this model (roughly 0.7 s/text measured earlier for RoBERTa-large).

## Measured performance (5-fold CV on REAL rows only, pooled out-of-fold)
Accuracy {metrics['real_accuracy']:.3f}, macro-F1 {metrics['real_macro_f1']:.3f}.
| Category | Precision | Recall | F1 | Real test rows |
|---|---|---|---|---|
{rows}

## Limitations (read before trusting)
- Rare categories have very few REAL examples (Forced Action 4, Sneaking 12, Obstruction 27); the rest of their training data is synthetic. Their scores are low-support estimates.
- The source labels are inconsistent for consent text (near-identical sentences appear as both Forced Action and Not Dark Pattern).
- Trained on short text fragments; unseen phrasings can fail. Text-only: cannot see visual/positional context (e.g. Misdirection by layout).
- Scareware / fake virus warnings are NOT in the taxonomy this model was trained on.
- The final model was trained on all data, so it has no held-out score of its own; cite the CV numbers above.
See evaluation/ for the full training data and out-of-fold predictions.
"""
    open(out_dir + "/MODEL_CARD.md", "w").write(card)
    import shutil; shutil.make_archive("darkguard_roberta_large_multiclass_v3", "zip", out_dir)
    print("\nSaved: darkguard_roberta_large_multiclass_v3.zip")
    try:
        from google.colab import files; files.download("darkguard_roberta_large_multiclass_v3.zip")
    except Exception as e: print("(download manually from the Files panel)", e)

if __name__ == "__main__":
    main()

torch 2.11.0+cu130 | transformers 5.18.0 | GPU: Tesla T4
Pool: 2640 rows = 2355 real (all kept) + 285 synthetic (0 synthetic dropped as duplicates) | OOD/blind holdout: 41 (1 removed: already in training)

Category x source:
source            p1_negative  p2v1_rare  p2v2_hard_negative  p2v2_rare  p2v3_hard_negative  real   All
Pattern Category                                                                                       
Forced Action               0         20                   0         20                   0     4    44
Misdirection                0          0                   0          0                   0   195   195
Not Dark Pattern          105          0                  34          0                  37  1177  1353
Obstruction                 0         15                   0         14                   0    27    56
Scarcity                    0          0                   0          0                   0   418   418
Sneaking                    0         20       

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

{'loss': '3.12', 'grad_norm': '51.49', 'learning_rate': '1.705e-05', 'epoch': '0.5952'}
{'loss': '0.9408', 'grad_norm': '54.38', 'learning_rate': '1.408e-05', 'epoch': '1.19'}
{'loss': '0.4966', 'grad_norm': '51.62', 'learning_rate': '1.11e-05', 'epoch': '1.786'}
{'loss': '0.3884', 'grad_norm': '1.551', 'learning_rate': '8.125e-06', 'epoch': '2.381'}
{'loss': '0.2931', 'grad_norm': '25.62', 'learning_rate': '5.149e-06', 'epoch': '2.976'}
{'loss': '0.1346', 'grad_norm': '8.29', 'learning_rate': '2.173e-06', 'epoch': '3.571'}
{'train_runtime': '174.6', 'train_samples_per_second': '61.43', 'train_steps_per_second': '3.85', 'train_loss': '0.8098', 'epoch': '4'}

Final-model sanity check (attempt 1): 6/6 passed
  OK   expected=Not Dark Pattern predicted=Not Dark Pattern conf=0.99 | Add to cart
  OK   expected=Not Dark Pattern predicted=Not Dark Pattern conf=1.00 | Checkout
  OK   expected=Not Dark Pattern predicted=Not Dark Pattern conf=1.00 | Thank you for your order.
  OK   expected=Scarc

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Saved: darkguard_roberta_large_multiclass_v3.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>